# Working with MOS target files

## Learning goals

The goal of this tutorial is to introduce you to the MOS target files, which contain information about the targets observed with the SDSS-V Multi-Object Spectrograph (MOS) spectrographs (i.e., APOGEE and BOSS). In particular, this tutorial uses the FITS version of the MOS target files. By the end of this tutorial, you should be able to:

- Understand what the MOS target files are and what information they contain.
- Access the MOS target files using the `sdss-access` package.
- Read the MOS target FITS files using `astropy`, including how to handle catalogues that have been partitioned into multiple files.

## What are the MOS target files?

SDSS-V target selection is a complex process that involves cross-matching multiple input catalogues and then applying various sets of selection criteria to identify targets that are suitable for observation with the SDSS-V MOS spectrographs. Each set of selection criteria is referred to as a "carton" and it defines a target class of objects that are of interest for a specific SDSS-V science program (e.g., RM, AQMES, Galactic Genesis, etc.) The input catalogues, results of the cross-match, and carton outputs, are stored in a PostgreSQL database at University of Utah. The contents of the database are also accessible to the public through the Catalog Archive Server (CAS) for each data release. More information about the SDSS-V target selection process can be found [here](https://www.sdss.org/dr20/targeting/).

The MOS target files contain the same information as the target selection database and are named for each one of the tables in the database. For example, the `catalogdb.catalog` table (`mos_catalog` in CAS) contains the `catalogid` identifiers for each unique target for a specific cross-match. This table is released as the [mos_target_catalog](https://data.sdss.org/datamodel/files/MOS_TARGET/V_TARG/mos_target_catalog.html) file product. The list of all MOS target products can be found in a table at the bottom of [this page](https://sdss.org/dr20/targeting/cross-match/).

For each table/MOS target product we provide two types of files: binary FITS tables and [Parquet files](https://parquet.apache.org). We recommend using [Parquet files](#using-parquet-files) as they are smaller, more efficient to read, and they can be processed in such a way that you can handle data that is larger than your computer's memory. To bypass that problem, the FITS files for large tables are partitioned into multiple files, which can be read one at a time. This tutorial uses the FITS files. We recommend reading the `mos_target_parquet.ipynb` notebook first, which includes examples of how to read the Parquet files.

## Dependencies

This tutorial makes use of the following Python packages:

- `sdss-access` to access SDSS data.
- `astropy` to read FITS files.

These packages can be installed using `pip`:

```bash
pip install sdss_access astropy
```


## Which files should I use?

These are our recommendations when querying the MOS targeting data:

- If you can, use [CASJobs](https://casjobs.sdss.org/) to query the `mos_` tables. Those tables have indices that will make most queries faster, and you don't need to worry about downloading large files or making sure your computer has enough memory.
- If CASJobs is not an option, use the Parquet files and `polars` to read them. With Parquet files you don't need to worry about chunking, and `polars` will allow you to perform operations on the data without having to load it all in memory.
- Use FITS files only as a last resource if you are using a programming language that doesn't have good support for Parquet files (e.g., IDL) or if you want to use the files with a software that only accepts FITS files.


## Using FITS files

In this section we will show how to work with the MOS target FITS files. As mentioned above, we do not recommend using these files unless you are using a programming language that doesn't have good support for Parquet files (in which case a Python tutorial may not be that useful!)

The MOS target FITS files contain the same information as the Parquet files in the form of binary FITS tables. However, to ensure that each file can be read in memory, large tables are partitioned into multiple files. For example, the `mos_target_sdss_id_flat` table is partitioned in 140 files, `mos_sdss_id_flat-001.fits` to `mos_sdss_id_flat-140.fits`. Let's write a function to retrieve the file paths. FITS files are also less efficient storing data than Parquet, so the total size of the FITS files for this example is significantly larger than the Parquet files.

<div class="alert alert-block alert-warning">
<b>Warning:</b> This code expects the code to exist locally and will not download any data. We recommend running this notebook in SciServer. If you are sure that you want to download the data locally, set the <code>download</code> parameter to <code>True</code>.
</div>


In [1]:
from sdss_access import Access, Path

import os
import pathlib

# For DR20, the targeting version is 2.0.0.
V_TARG = "2.0.0"

# If necessary, set the SAS_BASE_PATH environment variable.
os.environ["SAS_BASE_DIR"] = "/uufs/chpc.utah.edu/common/home/sdss50/"


def get_mos_target_fits_path(mos_target_product, download=False):
    """Get the path to the FITS file for a given MOS target product, downloading it if it doesn't exist."""

    path = Path(release="DR20")

    # Get the local SAS path to the FITS file for the specified MOS target product.
    mos_target_path = path.full(mos_target_product, v_targ=V_TARG, ftype="fits", num="*")

    # If the file contains a * it means it's partitioned. We do a bit of globbing to get all the files.
    if "*" in mos_target_path:
        mos_target_paths = pathlib.Path(mos_target_path).parent.glob(pathlib.Path(mos_target_path).name)
        mos_target_paths = list(sorted(map(str, mos_target_paths)))
    else:
        mos_target_paths = [mos_target_path]

    # Check that all the files exist.
    if not all(pathlib.Path(file).exists() for file in mos_target_paths):
        if download:
            access = Access(release="DR20")
            access.remote()
            access.add(mos_target_product, v_targ=V_TARG, ftype="fits", num="*")
            access.set_stream()
            access.commit()
        else:
            raise FileNotFoundError(f"One or more files for {mos_target_product!r} do not exist locally.")

    return mos_target_paths

This looks pretty similar to the Parquet function but note that we are adding `num="*"` to retrieve all the chunk files for each table. Let's see the result for the `mos_target_sdss_id_flat` and `mos_carton` tables. The first one will be a list of multiple files, with the second containing a single file because the `carton` table is small and doesn't need to be partitioned.


In [2]:
mos_carton_files = get_mos_target_fits_path("mos_target_carton", download=False)
mos_carton_to_target_files = get_mos_target_fits_path("mos_target_carton_to_target", download=False)
mos_target_files = get_mos_target_fits_path("mos_target_target", download=False)
mos_sdss_id_flat_files = get_mos_target_fits_path("mos_target_sdss_id_flat", download=False)
mos_catalog_to_gaia_dr3_files = get_mos_target_fits_path("mos_target_catalog_to_gaia_dr3_source", download=False)
mos_gaia_dr3_files = get_mos_target_fits_path("mos_target_gaia_dr3_source", download=False)

print(mos_sdss_id_flat_files)
print(mos_carton_files)


['/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-001.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-002.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-003.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-004.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-005.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-006.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-007.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-008.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-009.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/target/2.0.0/fits/mos_sdss_id_flat-010.fits', '/uufs/chpc.utah.edu/common/home/sdss50/dr20/mos/

There are two main ways to handle the partitioned files: we can either read each of them sequentially and concatenate the results (e.g., with the `astropy.table` `vstack` function), or we can read them one at a time and perform the necessary operations on each file before moving to the next one, saving the results we are interested in. The first approach is a bit less convoluted but it may exceed the available memory if the resulting table is too large. Here we will take the second approach.

We'll start by reading the `mos_carton` file and getting the `carton_pk` identifiers for the Galactic Genesis cartons.


In [3]:
import numpy
from astropy.table import Table

# Read the carton table and print a sample.
carton_table = Table.read(mos_carton_files[0], format="fits")
print(carton_table[:10])

       carton       carton_pk mapper_pk ... program  target_selection_plan
------------------- --------- --------- ... -------- ---------------------
      mwm_snc_100pc       126         0 ...  mwm_snc                 0.1.0
      mwm_snc_250pc       127         0 ...  mwm_snc                 0.1.0
       mwm_cb_300pc       128         0 ...   mwm_cb                 0.1.0
mwm_cb_cvcandidates       134         0 ...   mwm_cb                 0.1.0
        mwm_halo_sm       140         0 ... mwm_halo                 0.1.0
        mwm_halo_bb       143         0 ... mwm_halo                 0.1.0
         mwm_yso_s1       144         0 ...  mwm_yso                 0.1.0
         mwm_yso_s2       145         0 ...  mwm_yso                 0.1.0
       mwm_yso_s2-5       146         0 ...  mwm_yso                 0.1.0
         mwm_yso_s3       147         0 ...  mwm_yso                 0.1.0


In [4]:
# This file species is a single file, so we can do normal slicing on it.
# But we want to get any carton that starts with "mwm_galactic_core", which
# is not totally trivial. Since the table is small, we will simply loop over the rows.
carton_pks = []
for row in carton_table:
    if row["carton"].startswith("mwm_galactic_core"):
        carton_pks.append(int(row["carton_pk"]))

print(f"Carton PKs: {carton_pks}")

Carton PKs: [544, 1628, 1810, 1811]


In [5]:
from astropy.table import vstack, unique

# We will now do the same with the files in the mos_target_carton_to_target table/
# Here we iterate over each chunk file, get the rows that match the carton PKs,
# and keep only the target_pk column.
target_pk_chunks = []
for carton_to_target_file in mos_carton_to_target_files:
    carton_to_target_table = Table.read(carton_to_target_file, format="fits")

    # Get the rows that match the carton PKs and keep only the target_pk column.
    matching_rows = carton_to_target_table[numpy.isin(carton_to_target_table["carton_pk"], carton_pks)]
    target_pks = matching_rows["target_pk"]

    # Add these partial results to the list of target PK chunks.
    target_pk_chunks.append(target_pks)

# Concatenate the results.
carton_to_target_gal_gen = vstack(target_pk_chunks)

# Many target_pks will be repeated, so we get the unique values.
carton_to_target_gal_gen = unique(carton_to_target_gal_gen)

# Print results
print(carton_to_target_gal_gen)

target_pk
---------
246626552
246628288
246629026
246650074
246650089
246650110
246650860
246655241
246655333
246655420
      ...
365440161
365440162
365440163
365440164
365440165
365440166
365440167
365440168
365440169
365440170
Length = 11461373 rows


We will do the same to get the `catalogids` and the `sdss_ids`.

<div class="alert alert-block alert-warning">
<b>Warning:</b> The following cell may take over an hour to run! You can uncomment the `if len(...)` lines to test the code with a smaller number of files. Note that this will not give you the full results, but it will allow you to check that the code is working as expected without having to wait for all files to be processed.
</div>


In [6]:
from astropy.table import join

# Get the catalogids from the mos_target table
catalogid_chunks = []
for target_file in mos_target_files:
    target_table = Table.read(target_file, format="fits")

    # Get the rows that match the target PKs and keep only the catalogid column.
    # We will us the join function from astropy.table which is likely to be more
    # efficient.
    matching_rows = join(
        target_table,
        carton_to_target_gal_gen,
        keys_left=["target_pk"],
        keys_right=["target_pk"],
        join_type="inner",
    )
    catalogids = matching_rows["catalogid"]

    # Add these partial results to the list of catalogid chunks.
    if len(catalogids) > 0:
        catalogid_chunks.append(catalogids)

    # Uncomment the following lines if you want to stop after the
    # first file with results (for testing purposes).
    if len(catalogid_chunks) > 0:
        break

# Concatenate the results and get only unique values.
catalogids = unique(vstack(catalogid_chunks))

# Now do the same to get the associated sdss_ids from the mos_target_sdss_id_flat table.
sdss_id_chunks = []
for sdss_id_file in mos_sdss_id_flat_files:
    sdss_id_table = Table.read(sdss_id_file, format="fits")

    matching_rows = sdss_id_table[numpy.isin(sdss_id_table["catalogid"], catalogids["catalogid"])]
    sdss_ids = matching_rows[["sdss_id", "catalogid"]]

    if len(sdss_ids) > 0:
        sdss_id_chunks.append(sdss_ids)

    # Uncomment the following lines if you want to stop after the
    # first file with results (for testing purposes).
    if len(sdss_id_chunks) > 0:
        break

sdss_ids = unique(vstack(sdss_id_chunks))

print(sdss_ids)

 sdss_id      catalogid    
--------- -----------------
115413196 27021598066708077


We will not show how to get the Galactic coordinates for these targets using the Gaia DR3 tables, but the process would be similar to what we just did but using the `mos_target_gaia_dr3_source` and `mos_target_catalog_to_gaia_dr3_source`.

One important thing to note is the very different performance of the operations with FITS files compared to Parquet files. The operation we just performed using FITS files (getting all the `sdss_ids` associated with the Galactic Genesis cartons) took over two hours while the same operations with Parquet files took approximately 30 seconds.
